# Brighter-fatter kernel figure

Contact author: Alex Broughton
<br>Date: 


In [ ]:
! eups list -s | grep lsst_distrib

## Introduction

This notebook plots the vertical and horizontal brighter-fatter correction kernels derived from the electrostatic BFE model for one LSSTCam amplifier.

The output PDF is written to the current directory and copied into figures/ for the technote.


## 1.0 Set Up

In [ ]:
### Import packages and configure plotting defaults
from lsst.daf.butler import Butler
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import SymLogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

plt.rcParams.update({"font.size": 12})
import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
})


butler = Butler("main")

# Example amplifier used in the technote (detector 94, amp C03)
DETECTOR = 94
AMP = "C03"


## 2.0 Load calibration product


In [ ]:
ebf = butler.get(
    "electroBfDistortionMatrix",
    instrument="LSSTCam",
    detector=DETECTOR,
    collections="LSSTCam/calib",
)


## 3.0 Build convolution kernels

Tile the per-boundary shift matrices into charge-conserving vertical and horizontal kernels.


In [ ]:
def compute_k(electroBfDistortionMatrix):
    """Return vertical (kV) and horizontal (kH) brighter-fatter kernels."""
    r = electroBfDistortionMatrix.fitRange - 1
    aN = electroBfDistortionMatrix.aN
    aS = electroBfDistortionMatrix.aS
    aE = electroBfDistortionMatrix.aE
    aW = electroBfDistortionMatrix.aW

    kN = np.zeros((2 * r + 1, 2 * r + 1))
    kE = np.zeros_like(kN)

    # Quadrants for the north/south boundary-shift matrices
    kN[r:, r:] = aN
    kN[: r + 1, r:] = np.flipud(aN)
    kN[r:, : r + 1] = np.fliplr(aS)
    kN[: r + 1, : r + 1] = np.flipud(np.fliplr(aS))

    # Quadrants for the east/west boundary-shift matrices
    kE[r:, r:] = aE
    kE[: r + 1, r:] = np.flipud(aW)
    kE[r:, : r + 1] = np.fliplr(aE)
    kE[: r + 1, : r + 1] = np.flipud(np.fliplr(aW))

    # Enforce the sum-to-zero boundary condition
    kN[:, 0] = -kN[:, -1]
    kE[0, :] = -kE[-1, :]

    # Image axes are (parallel, serial), so transpose before returning
    return kN.T, kE.T


def sci_tex(x, sig=2):
    """Format a value as LaTeX scientific notation for colorbar labels."""
    if x == 0 or np.isclose(x, 0):
        return r"$0$"
    mant, exp = f"{x:.{sig}e}".split("e")
    return rf"${float(mant):.{sig - 1}f}\times 10^{{{int(exp)}}}$"


## 4.0 Plot kernels


In [ ]:
linthresh = 5e-8
cmap = "bwr"

kV, kH = compute_k(ebf)

vmax = np.nanmax([np.abs(kV).max(), np.abs(kH).max()])
vmin = -vmax
norm = SymLogNorm(linthresh=linthresh, vmin=vmin, vmax=vmax)
ticks = [vmin, -linthresh, 0.0, linthresh, vmax]
ticklabels = [
    rf"${sci_tex(vmin, 2)[1:-1]}$",
    rf"$-{sci_tex(linthresh, 2)[1:-1]}$",
    r"$0$",
    rf"${sci_tex(linthresh, 2)[1:-1]}$",
    rf"${sci_tex(vmax, 2)[1:-1]}$",
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

im1 = ax1.imshow(kV, origin="lower", cmap=cmap, norm=norm)
cax1 = make_axes_locatable(ax1).append_axes("right", size="5%", pad=0.15)
cb1 = fig.colorbar(im1, cax=cax1, ticks=ticks)
cb1.set_ticklabels(ticklabels)

im2 = ax2.imshow(kH, origin="lower", cmap=cmap, norm=norm)
cax2 = make_axes_locatable(ax2).append_axes("right", size="5%", pad=0.15)
cb2 = fig.colorbar(im2, cax=cax2, ticks=ticks)
cb2.set_ticklabels(ticklabels)

for ax in (ax1, ax2):
    ax.set_xticks(np.linspace(0, 19, 20))
    ax.set_yticks(np.linspace(0, 19, 20))

ax1.set_title(r"$K^V$")
ax2.set_title(r"$K^H$")
plt.tight_layout()
plt.savefig("ebf-det94-ampC03-kVH.pdf", format="pdf", bbox_inches="tight")
